<div style="text-align: center; width: 100%; padding: 2vw 0;">

# An Analysis of Clash Royale Cards and Decks

## DATA 11900

#### Ciaran Foglino, Aiden Gu, Violet Reed

<img src="./images/title.webp" style="width: 60%; margin-top: 2vw;">

</div>

## What is Clash Royale

- Matches take place on a map split by a river. Each player starts with three towers: one central King Tower and two Princess Towers.

<center>
<img src="./images/game.png" style="width: 10%; margin-top: 2vw;">
</center>

- Win by destroying more of your opponent's towers than they destroy of yours, or by destroying the King Tower for an instant victory.

- Fight using a custom deck of 8 cards. Cards are deployed using Elixir, which automatically refills over time during the match.

### What makes a good deck?

<img src="./images/hog2.6.webp" style="width: 40%; margin-top: 2vw;">

## Dataset Introduction and Acquisition

- Card data of all 126 cards is catalogued on the official *Clash Royale API*.

- Card usage rate and win rate comes from external sources like *RoyaleAPI*.

- The top 30 most successful decks also come from *RoyaleAPI*.

## Data Cleaning and EDA

Some null values observed in evolution level, damage, and health

- Cards without an evolution level were imported as `NaN` values and had to be recoded.

- Some card types (spells) were imported as `NaN` for health because they lacked any.

- Most missing damage values were corrected manually using game knowledge.

<center>
    <img src="./graphs/card_damage_vs_health.png" style="width: 60%; margin-top: 2vw;">
</center>

### Cards with high damage rarely have high health.

<center>
    <img src="./graphs/card_types.png" style="width: 60%; margin-top: 2vw;">
</center>

### Most cards are troop-type, with building and spell cards being rarer.

### Create dummy variables:

- `rarity` could be mapped to a 0–5 scale.

-  `wild`, `type`, `mobility`, `targets`, `attack_type`, and `groupCard` were one-hot encoded.

## Logistic Regression
<br>

### Data preparation

- Target variable: `Win Rate`, which ranges from 0 to 1.

- Sklearn's `LogisticRegression()` only allows binary target variable, so a median win rate of $53.435%$ was used as threshold to create a binary target variable.

- Cards above median labeled as 1 (high win rate), below as 0 (low win rate).

- Dataset was balanced with 51 cards in each class.

- Categorical variables converted to dummy variables (`wild`, `type`, `mobility`, `targets`, `attack_type`, `groupCard`).

- Numerical features (`elixirCost`, `damage11`, `hitpoints11`) scaled using `StandardScaler()`.

### Model Training and Testings

- Data split into $70%$ training and $30%$ testing.

- Accuracy on test set: $64.5%$.

- ROC-AUC score: $0.637$.

- 5-fold cross-validation accuracy: $61.7%$ ($\pm10.2%$).

|             | Predicted Win | Predicted Loss |
| --------:   | :-------:     | :------------: |
| Actual Win  | 10            | 3              |
| ACtual Loss | 8             | 10             |

### Feature Coefficients

- Positive coefficients: `mobility_flying` ($+0.172$), `wild_hero` ($+0.053$), `usage` ($+0.032$).

- Negative coefficients: `targets_both` ($-0.250$), `wild_evo` ($-0.206$), `type_building` ($-0.205$).

- Only `targets_both` reached statistical significance at $\alpha=0.05$

## Card Clustering
<br>

### Choosing Hyperparameters

- K-prototype was used instead of K-means because data contain a mix of categorical and numeric columns.

- 4 was chosen as the number of clusters for troop cards based on the elbow method.

<center>
    <img src="./graphs/kprototypes_elbow.png" style="width: 60%; margin-top: 2vw;">
</center>

- Buildings and spells were given their own clusters separate from the model, thus 6 clusters in total.

### Interpreting the Clusters

In [1]:
from IPython.display import IFrame
IFrame(src='./graphs/3d_scatter_plot_of_card_clusters.html', width='100%', height='550px')

- Cluster 0: `DPS` cards (high-damage, low-health, medium-cost), used to deal heavy damage.

- Cluster 1: `mini-tanks` (low-damage, medium-health, medium-cost), used to support other troops.

- Cluster 2: `tanks` (low-damage, high-health, high-cost), used to attack towers and build a push.

- Cluster 3: `cycle` cards (low-damage, low-health, low-cost), used to expend quickly to rotate to better cards.

- Cluster 4: `buildings`, stationary cards used mostly on defense.

- Cluster 5: `spells`, can be placed anywhere on the arena to deal damage.

## Deck Analysis

To analyze the synergy between card types, map the cards from the top 30 decks to their respective clusters and examine the ratios of clusters.

In [2]:
IFrame(src='./graphs/card_type_ratio.html', width='100%', height='900px')

### Hierarchical Clustering

<center>
    <img src="./graphs/deck_clustermap.png" style="width: 60%; margin-top: 2vw;">
</center>

### Correlation Matrix

<center>
    <img src="./graphs/card_type_correlation.png" style="width: 60%; margin-top: 2vw;">
</center>

- Mini-tanks and tanks appear together a lot, possibly in beat-down archetypes.

- Cycles and buildings also appear together, likely in cycle decks where buildings do a lot of the defending.

## The Average Deck

In [3]:
IFrame(src='./graphs/average_card_type_ratio.html', width='100%', height='600px')

| Cluster | Card Type |
| :-----: | --------- |
| 0 | DPS |
| 1 | Mini-tank |
| 2 | Tank |
| 3 | Cycle |
| 4 | Building |
| 5 | Spell |

## Sources

- Clash Royale API: [https://developer.clashroyale.com/](https://developer.clashroyale.com/#/)

- RoyaleAPI: [https://royaleapi.com/](https://royaleapi.com/)

- Kaggle: [https://www.kaggle.com/datasets/rodsaldanha/clash-royale-matches](https://www.kaggle.com/datasets/rodsaldanha/clash-royale-matches)

- Kaggle: [https://www.kaggle.com/datasets/niteshkakkar/clash-royal-cards-data](https://www.kaggle.com/datasets/niteshkakkar/clash-royal-cards-data)

- Kaggle: [https://www.kaggle.com/datasets/hrish4/clash-royale-cards-data](https://www.kaggle.com/datasets/hrish4/clash-royale-cards-data)